# 環境変数

In [ ]:
import os
os.environ["ERG_DATA_DIR"] = "/mnt/j/observation_data/"

# 3dfluxデータを時間軸に焼き直す

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

pt.del_data('*')

time_range_full = ['2017-11-15/16:10:00', '2017-11-15/16:30:00']
psp.erg.lepi(time_range_full, datatype='3dflux', get_support_data=True, no_update=True)

# 解析対象の狭い時間窓
time_range = ['2017-11-15/16:17:00', '2017-11-15/16:30:00']

# flux3d本体 (dims = time, v1(energy), v2(channel), v3(phase))  # [#/cm2/sr/sec/keV]
flux3d = pt.data_quants['erg_lepi_l2_3dflux_FPDU'].sel(
    time=slice(*time_range)
)

# 各軸の座標を取り出しておく
time_ax     = flux3d.time.values        # shape = (T,)
energy_ax   = flux3d.v1.values          # (E,)
channel_ax  = flux3d.v2.values          # (C,)
spin_ax     = flux3d.v3.values          # (S,)

time_num, energy_num, channel_num, spin_num = len(time_ax), len(energy_ax), len(channel_ax), len(spin_ax)   # T, E, C, S

In [ ]:
print(f'time_num = {time_num}, energy_num = {energy_num}, channel_num = {channel_num}, spin_num = {spin_num}')

In [ ]:
print(time_ax)

In [ ]:
# 1. 時刻オフセットをベクトル化で生成
# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

print(spin_offset_ns)
print(energy_offset_ns)

In [ ]:
offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

print(offset_ns)

In [ ]:
time_flat = (time_ax[:, None] + offset_ns[1][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)
print(time_flat[-40:])
print(time_ax[-2:])

In [ ]:
# 1. 時刻オフセットをベクトル化で生成
# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

# 2. flux3dをreshape
flux_E_TS_C = flux3d.transpose('v1_dim', 'time', 'v3_dim', 'v2_dim').values.reshape(energy_num, time_num*spin_num, channel_num)

# 3. xarray.DataArrayをエネルギーごとに生成
flux_data_arrays = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)

    flux_data_arrays[energy_i] = xr.DataArray(
        flux_E_TS_C[energy_i],
        dims=['time', 'channel'],
        coords={
            'time':         time_flat,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i]
        },
        attrs=flux3d.attrs,
        name=f'flux_energy_{energy_i}'
    )

In [ ]:
print(flux_data_arrays[0].time[:35])


sensor channel (CH0~CH7) & spin phase (spin time: 0~15) & timeから、各データに対応するpitch angleとgyrophaseをどのように与えるか？

まず、sensor channelとspin phaseから、SGA座標系における$\theta$(SGA-VZとのなす角)と$\varphi$(SGA-VX--SGA-VY平面上でのSGA-VXとのなす角)を導出

In [ ]:
fidu_angle_dict     = pt.data_quants['erg_lepi_l2_3dflux_FIDU_Angle_sga']
fidu_angle  = fidu_angle_dict['data'].astype(float)     # shape (2, 3, 16)
AZ_deg_mid = fidu_angle[0, 1, :channel_num]      # shape (C,)
AZ_deg_max = fidu_angle[0, 0, :channel_num]      # shape (C,)
AZ_deg_min = fidu_angle[0, 2, :channel_num]      # shape (C,)
print('AZ_deg_max = ', AZ_deg_max)
print('AZ_deg_mid = ', AZ_deg_mid)
print('AZ_deg_min = ', AZ_deg_min)

In [ ]:
#spin_phase_angle_deg_mid = (spin_ax * (360E0 / spin_num) - 90.) % 360.     # (S,)
#spin_phase_angle_deg_min = (spin_phase_angle_deg_mid - (360E0 / spin_num) / 2.) % 360.   # (S,)
#spin_phase_angle_deg_max = (spin_phase_angle_deg_mid + (360E0 / spin_num) / 2.) % 360.   # (S,)
#print('spin_phase_angle_deg_max = ', spin_phase_angle_deg_max)
#print('spin_phase_angle_deg_mid = ', spin_phase_angle_deg_mid)
#print('spin_phase_angle_deg_min = ', spin_phase_angle_deg_min)

In [ ]:
theta_sga = (90. - AZ_deg_mid) % 180.   # (C,)
#varphi_sga = (spin_phase_angle_deg_mid + 180.) % 360.   # (S,)
varphi_sga = -90. * np.ones(spin_num)   # (S,)

print('theta_sga = ', theta_sga)
print('varphi_sga = ', varphi_sga)

# (TxS, C)の2次元配列を生成
theta_sga_time = np.tile(theta_sga, (time_num*spin_num, 1))   # (TxS, C)
varphi_sga_times = np.tile(varphi_sga, (time_num, 1)).reshape(-1, 1)   # (TxS, 1)
varphi_sga_time = np.tile(varphi_sga_times, (1, channel_num))   # (TxS, C)
print('theta_sga_time.shape = ', theta_sga_time.shape)
print('varphi_sga_time.shape = ', varphi_sga_time.shape)
print('theta_sga_time = ', theta_sga_time)
print('varphi_sga_time = ', varphi_sga_time)

In [ ]:
angle_sga_time = np.stack((theta_sga_time, varphi_sga_time), axis=2)   # (TxS, C, 2)
print('angle_sga_time.shape = ', angle_sga_time.shape)
print('angle_sga_time = ', angle_sga_time)

vector_sga_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3)
vector_sga_time[:, :, 0] = np.sin(np.radians(angle_sga_time[:, :, 0])) * np.cos(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 1] = np.sin(np.radians(angle_sga_time[:, :, 0])) * np.sin(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 2] = np.cos(np.radians(angle_sga_time[:, :, 0]))
print('vector_sga_time.shape = ', vector_sga_time.shape)
print('vector_sga_time = ', vector_sga_time)

# 各チャンネルにベクトルを分解
vector_sga_time_ch = {}
for channel_i in range(channel_num):
    vector_sga_time_ch[channel_i] = xr.DataArray(
        vector_sga_time[:, channel_i, :],
        dims=['time', 'xyz'],
        coords={
            'time': time_flat,
            'xyz': ['x', 'y', 'z'],
            'channel': channel_ax[channel_i]
        },
        name=f'vector_sga_channel_{channel_i}'
    )
    print(f'channel {channel_i}: vector_sga_time_ch[{channel_i}] = ', vector_sga_time_ch[channel_i])

sga座標系をdsi座標系に変換する

In [ ]:
from pyspedas.projects.erg import erg_cotrans

# 各チャンネルのベクトルをdsi座標系に変換
vector_dsi_time_ch = {}
for channel_i in range(channel_num):
    pt.store_data(f'vector_sga_channel_{channel_i}', data={'x': time_flat, 'y': vector_sga_time_ch[channel_i].values})
    print(pt.data_quants[f'vector_sga_channel_{channel_i}'])
    # dsi座標系に変換
    erg_cotrans(f'vector_sga_channel_{channel_i}', f'vector_dsi_channel_{channel_i}', in_coord='sga', out_coord='dsi')
    vector_dsi_time_ch[channel_i] = xr.DataArray(
        pt.data_quants[f'vector_dsi_channel_{channel_i}'].values,
        dims=['time', 'xyz'],
        coords={
            'time': time_flat,
            'xyz': ['x', 'y', 'z'],
            'channel': channel_ax[channel_i]
        },
        name=f'vector_dsi_channel_{channel_i}'
    )
    print(f'channel {channel_i}: vector_dsi_time_ch[{channel_i}] = ', vector_dsi_time_ch[channel_i])

In [ ]:
#from pyspedas.projects.erg import erg_cotrans, sga2sgi
#
## 各チャンネルのベクトルをsgi座標系に変換
#vector_sgi_time_ch = {}
#for channel_i in range(channel_num):
#    pt.store_data(f'vector_sga_channel_{channel_i}', data={'x': time_flat, 'y': vector_sga_time_ch[channel_i].values})
#    print(pt.data_quants[f'vector_sga_channel_{channel_i}'])
#    # sgi座標系に変換
#    #erg_cotrans(f'vector_sga_channel_{channel_i}', f'vector_sgi_channel_{channel_i}', in_coord='sga', out_coord='sgi')
#    sga2sgi(f'vector_sga_channel_{channel_i}', f'vector_sgi_channel_{channel_i}')
#    vector_sgi_time_ch[channel_i] = xr.DataArray(
#        pt.data_quants[f'vector_sgi_channel_{channel_i}'].values,
#        dims=['time', 'xyz'],
#        coords={
#            'time': time_flat,
#            'xyz': ['x', 'y', 'z'],
#            'channel': channel_ax[channel_i]
#        },
#        name=f'vector_sgi_channel_{channel_i}'
#    )
#    print(f'channel {channel_i}: vector_sgi_time_ch[{channel_i}] = ', vector_sgi_time_ch[channel_i])

In [ ]:
#theta_sga = np.zeros((time_num*spin_num, channel_num))   # (TxS, C)
#varphi_sga = np.zeros((time_num*spin_num, channel_num))   # (TxS, C)
#for channel_i in range(channel_num):
#    theta_sga[:, channel_i] = np.degrees(np.arccos(vector_sga_time_ch[channel_i].sel(xyz='z').values))
#    phi = np.degrees(np.arctan2(vector_sga_time_ch[channel_i].sel(xyz='y').values, vector_sga_time_ch[channel_i].sel(xyz='x').values))  # -180 < phi <= 180
#    varphi_sga[:, channel_i] = (phi + 360.) % 360.   # 0 <= phi < 360
#
#theta_sgi = np.zeros((time_num*spin_num, channel_num))   # (TxS, C)
#varphi_sgi = np.zeros((time_num*spin_num, channel_num))   # (TxS, C)
#for channel_i in range(channel_num):
#    theta_sgi[:, channel_i] = np.degrees(np.arccos(vector_sgi_time_ch[channel_i].sel(xyz='z').values))
#    phi = np.degrees(np.arctan2(vector_sgi_time_ch[channel_i].sel(xyz='y').values, vector_sgi_time_ch[channel_i].sel(xyz='x').values))  # -180 < phi <= 180
#    varphi_sgi[:, channel_i] = (phi + 360.) % 360.   # 0 <= phi < 360
#
#print('theta_sga = ', theta_sga[100, :])
#print('theta_sgi = ', theta_sgi[100, :])
#print('varphi_sga = ', varphi_sga[800:816, 0])
#print('varphi_sgi = ', varphi_sgi[800:816, 0])

In [ ]:
#pt.data_quants['vector_sgi_channel_0']

In [ ]:
#from pyspedas.projects.erg import erg_cotrans, sgi2dsi
#
## 各チャンネルのベクトルをdsi座標系に変換
#vector_dsi_time_ch = {}
#for channel_i in range(channel_num):
#    pt.store_data(f'vector_sgi_channel_{channel_i}', data={'x': time_flat, 'y': vector_sgi_time_ch[channel_i].values})
#    print(pt.data_quants[f'vector_sgi_channel_{channel_i}'])
#    # dsi座標系に変換
#    #erg_cotrans(f'vector_sgi_channel_{channel_i}', f'vector_dsi_channel_{channel_i}', in_coord='sgi', out_coord='dsi')
#    sgi2dsi(f'vector_sgi_channel_{channel_i}', f'vector_dsi_channel_{channel_i}')
#    vector_dsi_time_ch[channel_i] = xr.DataArray(
#        pt.data_quants[f'vector_dsi_channel_{channel_i}'].values,
#        dims=['time', 'xyz'],
#        coords={
#            'time': time_flat,
#            'xyz': ['x', 'y', 'z'],
#            'channel': channel_ax[channel_i]
#        },
#        name=f'vector_dsi_channel_{channel_i}'
#    )
#    print(f'channel {channel_i}: vector_dsi_time_ch[{channel_i}] = ', vector_dsi_time_ch[channel_i])

In [ ]:
#vector_dsi_time_ch[0].time

In [ ]:
## 一度、DSI座標系でのθ, φを計算してみる
#theta_dsi = np.zeros((time_num*spin_num, channel_num))   # (TxS, C)
#phi_dsi = np.zeros((time_num*spin_num, channel_num))   # (TxS, C)
#for channel_i in range(channel_num):
#    x = vector_dsi_time_ch[channel_i].sel(xyz='x').values
#    y = vector_dsi_time_ch[channel_i].sel(xyz='y').values
#    z = vector_dsi_time_ch[channel_i].sel(xyz='z').values
#    r = np.sqrt(x**2 + y**2 + z**2)
#    theta_dsi[:, channel_i] = np.degrees(np.arccos(z / r))   # 0 <= theta <= 180
#    phi = np.degrees(np.arctan2(y, x))   # -180 < phi <= 180
#    phi_dsi[:, channel_i] = (phi + 360.) % 360.   # 0 <= phi < 360
#print('theta_sgi =', theta_sgi[100, :])
#print('theta_dsi = ', theta_dsi[100, :])
#print('varphi_sgi =', varphi_sgi[800:816, 0])
#print('phi_dsi = ', phi_dsi[800:816, 0])

dsi座標系をFAC座標系に変換する

In [ ]:
# まず、dsi座標系をFAC座標系に変換するための変換行列を計算
psp.erg.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='64hz', coord='dsi', no_update=True)

psp.erg.orb(trange=time_range_full, level='l2', datatype='def')

In [ ]:
B_64hz_dsi = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].sel(time=slice(*time_range_full)).sortby('time')
B_8sec_dsi = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].sel(time=slice(*time_range_full)).sortby('time')
print(B_64hz_dsi)


In [ ]:
import matplotlib.pyplot as plt

B_8sec_dsi_x, B_8sec_dsi_y, B_8sec_dsi_z = B_8sec_dsi.data[:, 0], B_8sec_dsi.data[:, 1], B_8sec_dsi.data[:, 2]
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax[0].plot(B_8sec_dsi.time, B_8sec_dsi_x)
ax[0].set_ylabel('B_x [nT]')
ax[0].minorticks_on()
ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
ax[1].plot(B_8sec_dsi.time, B_8sec_dsi_y)
ax[1].set_ylabel('B_y [nT]')
ax[1].minorticks_on()
ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
ax[2].plot(B_8sec_dsi.time, B_8sec_dsi_z)
ax[2].set_ylabel('B_z [nT]')
ax[2].set_xlabel('Time [UT]')
ax[2].minorticks_on()
ax[2].grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
#dt = (B_64hz_dsi.time[1] - B_64hz_dsi.time[0]).astype('timedelta64[ns]').astype('float') * 1E-9   # sec
#win_pts = int(8.0 / dt)
#B_64hz_dsi_avg = B_64hz_dsi.rolling(time=win_pts, center=True).mean('time')
#
#import matplotlib.pyplot as plt
#
#B_64hz_dsi_avg_x, B_64hz_dsi_avg_y, B_64hz_dsi_avg_z = B_64hz_dsi_avg.data[:, 0], B_64hz_dsi_avg.data[:, 1], B_64hz_dsi_avg.data[:, 2]
#fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
#ax[0].plot(B_64hz_dsi_avg.time, B_64hz_dsi_avg_x)
#ax[0].set_ylabel('B_x [nT]')
#ax[0].minorticks_on()
#ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
#ax[1].plot(B_64hz_dsi_avg.time, B_64hz_dsi_avg_y)
#ax[1].set_ylabel('B_y [nT]')
#ax[1].minorticks_on()
#ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
#ax[2].plot(B_64hz_dsi_avg.time, B_64hz_dsi_avg_z)
#ax[2].set_ylabel('B_z [nT]')
#ax[2].set_xlabel('Time [UT]')
#ax[2].minorticks_on()
#ax[2].grid(True, which='both', linestyle='--', alpha=0.5)
#plt.tight_layout()
#plt.show()

In [ ]:
#import xarray as xr
#import numpy as np
#
#def dsi_to_fac(vector_dsi: xr.DataArray, window_sec: float = 100.0) -> xr.DataArray:
#    """
#    dsi座標系のベクトルをFAC座標系に変換する。
#    背景磁場の計算には窓幅window_secの移動平均を用いる。
#    vector_dsi: xr.DataArray
#        dims = ['time', 'xyz']
#        coords = {
#            'time': 時刻 (np.datetime64)
#            'xyz': ['x', 'y', 'z']
#    """
#
#    vector_dsi = vector_dsi.sortby('time')
#
#    # 背景磁場を計算
#    dt = (B_64hz_dsi.time[1] - B_64hz_dsi.time[0]).astype('timedelta64[ns]').astype('float') * 1E-9   # sec
#    win_pts = int(window_sec / dt)
#    print(f'dt = {dt} sec, window_pts = {win_pts} pts for window_sec = {window_sec} sec')
#    B_dsi_mean = B_64hz_dsi.rolling(time=win_pts, center=True).mean('time')
#    B_dsi_mean = B_dsi_mean.interp(time=vector_dsi.time)   # vector_dsiの時刻に補間
#
#    # FAC基底ベクトルを計算
#    z0 = np.array([0.0, 0.0, 1.0])
#    e2 = np.cross(z0, B_dsi_mean)
#    e1 = np.cross(e2, B_dsi_mean)
#    e3 = B_dsi_mean
#
#    def _normalize(v):
#        return v / np.linalg.norm(v, axis=1, keepdims=True)
#    
#    e1_hat = _normalize(e1)
#    e2_hat = _normalize(e2)
#    e3_hat = _normalize(e3)
#
#    # 回転行列とベクトル変換
#    R = np.stack([e1_hat, e2_hat, e3_hat], axis=2)
#    vector_fac = np.einsum('tji,tj->ti', R, vector_dsi.values)
#
#    vector_fac_da = xr.DataArray(
#        vector_fac,
#        dims=['time', 'xyz'],
#        coords={
#            'time': vector_dsi.time,
#            'xyz': ['x', 'y', 'z']
#        },
#        name=vector_dsi.name.replace('dsi', 'fac')
#    )
#    return vector_fac_da, R

In [ ]:
#import xarray as xr
#import numpy as np
#
#def dsi_to_fac(vector_dsi: xr.DataArray, window_sec: float = 100.0) -> xr.DataArray:
#    """
#    dsi座標系のベクトルをFAC座標系に変換する。
#    背景磁場の計算には窓幅window_secの移動平均を用いる。
#    vector_dsi: xr.DataArray
#        dims = ['time', 'xyz']
#        coords = {
#            'time': 時刻 (np.datetime64)
#            'xyz': ['x', 'y', 'z']
#    """
#
#    vector_dsi = vector_dsi.sortby('time')
#
#    # 背景磁場を計算
#    B_dsi_mean = B_8sec_dsi.interp(time=vector_dsi.time)   # vector_dsiの時刻に補間
#
#    # FAC基底ベクトルを計算
#    z0 = np.array([0.0, 0.0, 1.0])
#    e2 = np.cross(z0, B_dsi_mean)
#    e1 = np.cross(e2, B_dsi_mean)
#    e3 = B_dsi_mean
#
#    def _normalize(v):
#        return v / np.linalg.norm(v, axis=1, keepdims=True)
#    
#    e1_hat = _normalize(e1)
#    e2_hat = _normalize(e2)
#    e3_hat = _normalize(e3)
#
#    # 回転行列とベクトル変換
#    R = np.stack([e1_hat, e2_hat, e3_hat], axis=2)
#    vector_fac = np.einsum('tji,tj->ti', R, vector_dsi.values)
#
#    vector_fac_da = xr.DataArray(
#        vector_fac,
#        dims=['time', 'xyz'],
#        coords={
#            'time': vector_dsi.time,
#            'xyz': ['x', 'y', 'z']
#        },
#        name=vector_dsi.name.replace('dsi', 'fac')
#    )
#    return vector_fac_da, R

In [ ]:
from pyspedas.projects.erg.satellite.erg.particle.erg_pgs_make_fac import erg_pgs_make_fac

times_mag = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].time.values
fac_mat_mag = erg_pgs_make_fac(times_mag, 'erg_mgf_l2_mag_8sec_dsi', 'erg_orb_l2_pos_gse', fac_type='mphism')


In [ ]:
from pytplot import store_data
from pyspedas import tvector_rotate

print(fac_mat_mag)

# 1) NaN を含む時刻を落とす
valid = np.isfinite(fac_mat_mag).all(axis=(1,2))
times_mag_v = times_mag[valid]
R = fac_mat_mag[valid].astype(float)                          # (M,3,3)

# 2) 直交化（各時刻ごとに最も近い回転行列へ射影：SVD）
#    R ≈ U @ V^T、det<0 のときは反転して det=+1 に補正
U, s, Vt = np.linalg.svd(R)
R_ortho = U @ Vt
neg = (np.linalg.det(R_ortho) < 0)
if np.any(neg):
    Vt[neg, 2, :] *= -1.0
    R_ortho = U @ Vt

# 3) tplot に保存 → ベクトル時刻（64 Hz）へ slerp で補間しつつ回転
store_data('fac_on_mag', data={'x': times_mag_v, 'y': R_ortho})
tvector_rotate('fac_on_mag', 'erg_mgf_l2_mag_64hz_dsi', newname='B_fac')

B_64hz_fac = pt.data_quants['B_fac']

B_64hz_fac_x, B_64hz_fac_y, B_64hz_fac_z = B_64hz_fac.sel(v_dim=0), B_64hz_fac.sel(v_dim=1), B_64hz_fac.sel(v_dim=2)

fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax[0].plot(B_64hz_fac.time, B_64hz_fac_x)
ax[0].set_ylabel('B_x [nT]')
ax[0].minorticks_on()
ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
ax[1].plot(B_64hz_fac.time, B_64hz_fac_y)
ax[1].set_ylabel('B_y [nT]')
ax[1].minorticks_on()
ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
ax[2].plot(B_64hz_fac.time, B_64hz_fac_z)
ax[2].set_ylabel('B_z [nT]')
ax[2].set_xlabel('Time')
ax[2].minorticks_on()
ax[2].grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

time_ax_range_min = time_ax[0]
time_ax_range_max = time_ax[0] + np.timedelta64(30, 's')
fig, ax = plt.subplots(figsize=(10, 6))
e3_z = R_ortho[:, 2, 2]
angle_deg = np.degrees(np.arccos(e3_z))
ax.plot(times_mag_v, angle_deg)
ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
ax.set_xlabel('Time')
ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlim(time_ax_range_min, time_ax_range_max)
ax.set_ylim(70.5, 70.8)
plt.tight_layout()
plt.show()

In [ ]:
#B_64hz_fac, R_tensor = dsi_to_fac(B_64hz_dsi, window_sec=8.0)
#print(B_64hz_fac)
#
#import matplotlib.pyplot as plt
#
#B_64hz_fac_x, B_64hz_fac_y, B_64hz_fac_z = B_64hz_fac.sel(xyz='x'), B_64hz_fac.sel(xyz='y'), B_64hz_fac.sel(xyz='z')
#fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
#ax[0].plot(B_64hz_fac.time, B_64hz_fac_x)
#ax[0].set_ylabel('B_x [nT]')
#ax[0].minorticks_on()
#ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
#ax[1].plot(B_64hz_fac.time, B_64hz_fac_y)
#ax[1].set_ylabel('B_y [nT]')
#ax[1].minorticks_on()
#ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
#ax[2].plot(B_64hz_fac.time, B_64hz_fac_z)
#ax[2].set_ylabel('B_z [nT]')
#ax[2].set_xlabel('Time')
#ax[2].minorticks_on()
#ax[2].grid(True, which='both', linestyle='--', alpha=0.5)
#plt.tight_layout()
#plt.show()
#
#time_ax_range_min = time_ax[0]
#time_ax_range_max = time_ax[0] + np.timedelta64(30, 's')
#fig, ax = plt.subplots(figsize=(10, 6))
#e3_z = R_tensor[:, 2, 2]
#angle_deg = np.degrees(np.arccos(e3_z))
#ax.plot(B_64hz_fac.time, angle_deg)
#ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
#ax.set_xlabel('Time')
#ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
#ax.minorticks_on()
#ax.grid(True, which='both', linestyle='--', alpha=0.5)
##ax.set_xlim(time_ax_range_min, time_ax_range_max)
##ax.set_ylim(70.5, 70.8)
#plt.tight_layout()
#plt.show()

In [ ]:
# 各チャンネルのベクトルをFAC座標系に変換
import matplotlib.pyplot as plt
vector_fac_time_ch = {}
for channel_i in range(channel_num):
    pt.store_data('vector_dsi_time', data={'x': vector_dsi_time_ch[channel_i].time, 'y': vector_dsi_time_ch[channel_i].data})
    psp.tvector_rotate('fac_on_mag', 'vector_dsi_time', newname='vector_fac_time')
    vector_fac_time_ch[channel_i] = pt.data_quants['vector_fac_time']
    print(f'channel {channel_i}: vector_fac_time_ch[{channel_i}] = ', vector_fac_time_ch[channel_i])

#    if channel_i != 0:
#        continue
#    time_ax_range_min = time_ax[0]
#    time_ax_range_max = time_ax[0] + np.timedelta64(30, 's')
#    fig, ax = plt.subplots(figsize=(10, 6))
#    e3_z = Ro_tensor[:, 2, 2]
#    angle_deg = np.degrees(np.arccos(e3_z))
#    ax.plot(vector_fac_time_ch[channel_i].time, angle_deg)
#    ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
#    ax.set_xlabel('Time')
#    ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
#    ax.minorticks_on()
#    ax.grid(True, which='both', linestyle='--', alpha=0.5)
#    #ax.set_xlim(time_ax_range_min, time_ax_range_max)
#    #ax.set_ylim(70.5, 70.8)
#    plt.tight_layout()
#    plt.show()

In [ ]:
# fac vectorをpitch angle, gyrophase angleに変換
vector_fac_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3) を生成
for channel_i in range(channel_num):
    vector_fac_time[:, channel_i, :] = vector_fac_time_ch[channel_i].values
print('vector_fac_time.shape = ', vector_fac_time.shape)
print('vector_fac_time = ', vector_fac_time)

# fac vectorからpitch angleとgyrophaseを計算
pitch_angle_deg_time = np.degrees(np.arccos(vector_fac_time[:, :, 2]))   # (TxS, C)
gyrophase_deg_time = (np.degrees(np.arctan2(vector_fac_time[:, :, 1], vector_fac_time[:, :, 0])) + 90.) % 360.   # (TxS, C)
print('pitch_angle_deg_time.shape = ', pitch_angle_deg_time.shape)
print('gyrophase_deg_time.shape = ', gyrophase_deg_time.shape)
print('pitch_angle_deg_time = ', pitch_angle_deg_time)
print('gyrophase_deg_time = ', gyrophase_deg_time)

In [ ]:
# 各エネルギーチャンネルについて、fluxとpitch angle, gyrophase angleを結合
flux_pitch_gyro_data_arrays = {}
for energy_i in range(energy_num):
    # (TxS, C, 3)の3次元配列を生成 (Channelはpitch angle & gyrophaseに含まれる)
    flux_pitch_gyro_data_arrays[energy_i] = xr.DataArray(
        np.stack((
            flux_E_TS_C[energy_i],                               # (TxS, C)
            pitch_angle_deg_time,              # (TxS, C)
            gyrophase_deg_time                 # (TxS, C)
        ), axis=2),                                               # (TxS, C, 3)
        dims=['time', 'channel', 'variable'],
        coords={
            'time':         flux_data_arrays[energy_i]['time'].values,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i],
            'variable':     ['flux', 'pitch_angle_deg', 'gyrophase_deg']
        },
        attrs=flux3d.attrs,
        name=f'flux_pitch_gyro_energy_{energy_i}'
    )
    print(f'energy {energy_i}: flux_pitch_gyro_data_arrays[{energy_i}]  = ', flux_pitch_gyro_data_arrays[energy_i])

# 横軸: 時間、縦軸: pitch angle/gyrophase(2つ), colorbar: fluxのplot

In [ ]:
time_ax_range_min = time_ax[0]
time_ax_range_max = time_ax[0] + np.timedelta64(5, 'm')

for count_i in range(energy_num):
    time_ax = flux_pitch_gyro_data_arrays[count_i]['time'].values
    time_ax_mesh, channel_ax_mesh = np.meshgrid(time_ax, channel_ax, indexing='ij')
    pitch_angle_data = flux_pitch_gyro_data_arrays[count_i].data[:, :, 1]
    gyrophase_data = flux_pitch_gyro_data_arrays[count_i].data[:, :, 2]
    flux_data = flux_pitch_gyro_data_arrays[count_i].data[:, :, 0]

    if count_i != 5:
        continue

    # 横軸: 時間、縦軸: pitch angle, colorbar: fluxのplot
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from matplotlib.colors import LogNorm 
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    # 横軸: time_ax_mesh, 縦軸: pitch_angle_data, colorbar: flux_data
    sc = ax0.scatter(
        time_ax_mesh.flatten(),
        pitch_angle_data.flatten(),
        c=flux_data.flatten(),
        s=10,
        cmap='turbo',
        norm=LogNorm()
        )
    ax0.set_ylabel('pitch angle (deg)')
    ax0.set_title(f'LEP-i flux (energy = {energy_ax[count_i]:.4f} keV)')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.minorticks_on()
    ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 180)
    ax0.set_yticks(np.arange(0, 181, 15))
    ax0.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(sc, ax=ax0, label='flux [#/cm2/sr/sec/keV]')
    ax1 = fig.add_subplot(212)
    # 横軸: time_ax_mesh, 縦軸: gyrophase_data, colorbar: flux_data
    sc = ax1.scatter(
        time_ax_mesh.flatten(),
        gyrophase_data.flatten(),
        c=flux_data.flatten(),
        s=10,
        cmap='turbo',
        norm=LogNorm()
        )
    ax1.set_ylabel('gyrophase (deg)')
    ax1.set_title(f'LEP-i flux (energy = {energy_ax[count_i]:.4f} keV)')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.minorticks_on()
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360)
    ax1.set_yticks(np.arange(0, 361, 30))
    ax1.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(sc, ax=ax1, label='flux [#/cm2/sr/sec/keV]')
    plt.tight_layout()
    plt.show()

## meshでplotしてみる

In [ ]:
import numpy as np
import xarray as xr
from scipy.stats import binned_statistic_2d
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import matplotlib.colors as mcolors

# 格子点を定義
pitch_angle_bins = np.linspace(0, 180, 17)    # 16 bins (0, 11.25, 22.5, ..., 180)
gyrophase_bins = np.linspace(0, 360, 17)       # 16 bins (0, 22.5, 45, ..., 360)
dt = 8.0   # sec
time_bins = np.arange(time_ax_range_min.astype('datetime64[s]').astype('float') + 4.0,
                      time_ax_range_max.astype('datetime64[s]').astype('float') + dt + 4.0,
                      dt)   # dt secごとのbin
time_bins = time_bins.astype('datetime64[s]')
print('time_bins = ', time_bins)

# ---- helper: datetime64 -> int (秒) ----
def to_sec_i64(t):
    t = np.asarray(t).astype('datetime64[s]')
    return t.astype('int64')  # epoch秒

def make_lognorm(arr, lo_q=1, hi_q=99, min_span=10.0):
    # 正の有限値だけ
    pos = arr[np.isfinite(arr) & (arr > 0)]
    if pos.size == 0:
        return None
    vmin = np.nanpercentile(pos, lo_q)
    vmax = np.nanpercentile(pos, hi_q)
    # 幅が潰れたら最低幅を確保
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin * min_span
    return mcolors.LogNorm(vmin=vmin, vmax=vmax)

# 時間ビン（秒の整数エッジに）
time_bins_sec = to_sec_i64(time_bins)

for energy_i in range(energy_num):
    time_ax = flux_pitch_gyro_data_arrays[energy_i]['time'].values
    pitch_angle_data = flux_pitch_gyro_data_arrays[energy_i].data[:, :, 1]
    gyrophase_data   = flux_pitch_gyro_data_arrays[energy_i].data[:, :, 2]
    flux_data        = flux_pitch_gyro_data_arrays[energy_i].data[:, :, 0]

    # 時刻を2Dに展開 → 秒の整数に
    time_ax_mesh = np.broadcast_to(time_ax[:, None], pitch_angle_data.shape)
    tx_sec = to_sec_i64(time_ax_mesh.ravel())

    pa = pitch_angle_data.ravel().astype(float)
    gy = gyrophase_data.ravel().astype(float)
    fv = flux_data.ravel().astype(float)

    good = np.isfinite(pa) & np.isfinite(gy) & np.isfinite(fv)
    tx_sec, pa, gy, fv = tx_sec[good], pa[good], gy[good], fv[good]

    # --- 2Dビニング（平均：欠損は無視） ---
    hist_pitch, xedges_pitch, yedges_pitch, _ = binned_statistic_2d(
        tx_sec, pa, fv, statistic=np.nanmean,
        bins=[time_bins_sec, pitch_angle_bins]
    )
    hist_gyro, xedges_gyro, yedges_gyro, _ = binned_statistic_2d(
        tx_sec, gy, fv, statistic=np.nanmean,
        bins=[time_bins_sec, gyrophase_bins]
    )

    # ビン中心 → 描画用に datetime64 へ戻す
    tcent_sec = (xedges_pitch[:-1] + xedges_pitch[1:]) // 2
    time_cent = tcent_sec.astype('datetime64[s]')
    pa_cent   = 0.5*(yedges_pitch[:-1] + yedges_pitch[1:])
    gy_cent   = 0.5*(yedges_gyro[:-1]  + yedges_gyro[1:])

    cnt_pitch, _, _ = np.histogram2d(tx_sec, pa, bins=[time_bins_sec, pitch_angle_bins])
    cnt_gyro, _, _  = np.histogram2d(tx_sec, gy, bins=[time_bins_sec, gyrophase_bins])
    hist_pitch[cnt_pitch == 0] = np.nan
    hist_gyro[cnt_gyro == 0]   = np.nan

    norm_pitch = mcolors.LogNorm(vmin=1E4, vmax=3E5)
    norm_gyro  = mcolors.LogNorm(vmin=7E4, vmax=3E5)

    if norm_pitch is None or norm_gyro is None:
        print(f'energy {energy_i}: no positive flux data')
        continue

    if energy_i != 5:
        continue

    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    pcm = ax0.pcolormesh(
        time_cent, pa_cent, hist_pitch.T,
        cmap='turbo', norm=norm_pitch,
        shading='auto'
    )
    ax0.set_ylabel('pitch angle (deg)')
    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 180); ax0.set_yticks(np.arange(0, 181, 15))
    ax0.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(pcm, ax=ax0, label='flux [#/cm2/sr/sec/keV]')

    ax1 = fig.add_subplot(212)
    pcm = ax1.pcolormesh(
        time_cent, gy_cent, hist_gyro.T,
        cmap='turbo', norm=norm_gyro,
        shading='auto'
    )
    ax1.set_ylabel('gyrophase (deg)')
    ax1.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360); ax1.set_yticks(np.arange(0, 361, 30))
    ax1.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(pcm, ax=ax1, label='flux [#/cm2/sr/sec/keV]')
    plt.tight_layout(); plt.show()


## erg_lep_part_productsによるpa分布

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

psp.erg.lepi(trange=time_range_full, level='l2', datatype='3dflux', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', no_update=True)
psp.erg.orb(trange=time_range_full, level='l2', datatype='def', no_update=True)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['pa', 'gyro'],
    pitch=[0, 180],
    energy=[8E3, 9E3],
    mag_name='erg_mgf_l2_mag_8sec_dsi',
    pos_name='erg_orb_l2_pos_gse'
)

erg_lepi_pa = pt.data_quants['erg_lepi_l2_3dflux_FPDU_pa']
erg_lepi_gyro = pt.data_quants['erg_lepi_l2_3dflux_FPDU_gyro']
print(erg_lepi_pa)
print(erg_lepi_gyro)

In [ ]:
vars = ['erg_lepi_l2_3dflux_FPDU_pa', 'erg_lepi_l2_3dflux_FPDU_gyro']
pt.options(vars, 'colormap', 'turbo')
pt.options('erg_lepi_l2_3dflux_FPDU_pa', 'z_range', [1E1, 3E2])
pt.options('erg_lepi_l2_3dflux_FPDU_gyro', 'z_range', [7E1, 3E2])
pt.timespan('2017-11-15/16:17:00', 5, keyword='minute')
pt.tplot(vars)

# $\bf{B}_{\mathrm{w}}$の位相$\psi_{\mathrm{B}}$を求める。

# まず、FAC座標系の磁場の波形を求める。

In [ ]:
B_64hz_fac = pt.data_quants['B_fac']
print(B_64hz_fac)

import matplotlib.pyplot as plt

B_64hz_fac_x, B_64hz_fac_y, B_64hz_fac_z = B_64hz_fac.sel(v_dim=0), B_64hz_fac.sel(v_dim=1), B_64hz_fac.sel(v_dim=2)
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax[0].plot(B_64hz_fac.time, B_64hz_fac_x)
ax[0].set_ylabel('B_x [nT]')
ax[0].minorticks_on()
ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
ax[1].plot(B_64hz_fac.time, B_64hz_fac_y)
ax[1].set_ylabel('B_y [nT]')
ax[1].minorticks_on()
ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
ax[2].plot(B_64hz_fac.time, B_64hz_fac_z)
ax[2].set_ylabel('B_z [nT]')
ax[2].set_xlabel('Time')
ax[2].minorticks_on()
ax[2].grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

#time_ax_range_min = time_ax[0]
#time_ax_range_max = time_ax[0] + np.timedelta64(30, 's')
#fig, ax = plt.subplots(figsize=(10, 6))
#e3_z = R_tensor[:, 2, 2]
#angle_deg = np.degrees(np.arccos(e3_z))
#ax.plot(B_64hz_fac.time, angle_deg)
#ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
#ax.set_xlabel('Time')
#ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
#ax.minorticks_on()
#ax.grid(True, which='both', linestyle='--', alpha=0.5)
##ax.set_xlim(time_ax_range_min, time_ax_range_max)
##ax.set_ylim(70.5, 70.8)
#plt.tight_layout()
#plt.show()

## $0.6 \, \mathrm{Hz} < f < 0.75 \, \mathrm{Hz}$でバンドパスフィルタをかける。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 64.                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth（2-pole × 2-stage）

# 0.6 Hzから0.75 Hzの範囲のみを通すband-passフィルタ
lowcut = 0.6
highcut = 0.75

# btypeを'bandpass'に設定
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

# ------------------ インパルス応答 (変更なし) ------------------
n = 2048
delta = np.zeros(n)
delta[n//2] = 1

h = sosfiltfilt(sos, delta)
t = (np.arange(n) - n//2) / fs

# ------------------ 周波数応答 (変更なし) ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)
H_dbl = np.abs(H)**2

# ------------------ プロット (タイトルとハイライトを変更) ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse Response (Band-pass filter)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)| (double-pass)')

# 除去帯域を半透明のグレーで示す
axs[1].axvspan(lowcut, highcut, color='gray', alpha=0.3, label=f'Stop band ({lowcut:.2f}-{highcut:.2f} Hz)')

axs[1].set_title('Magnitude Response (Band-pass filter)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20, 10)
axs[1].set_xlim(3E-1, 1)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく


# フィルタパラメータ
fs = 64.                  # サンプリング周波数 [Hz]
lowcut = 0.6              # 通過域の下限周波数 [Hz]
highcut = 0.75            # 通過域の上限周波数 [Hz]
order = 4                 # フィルタの次数
window_sec = 100.0        # 背景磁場の移動平均窓幅 [sec]

sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

B_64hz_fac_bandpass = np.zeros(B_64hz_fac.data.shape) * np.nan  # NaNで初期化
for i in range(3):
    B_64hz_fac_bandpass[:, i] = apply_filter_segmented(B_64hz_fac.data[:, i], sos)
B_64hz_fac_bandpass_da = xr.DataArray(
    B_64hz_fac_bandpass,
    dims=B_64hz_fac.dims,
    coords=B_64hz_fac.coords,
    name='erg_mgf_l2_mag_64hz_fac_bandpass'
)


In [ ]:
time_ax_range_min = time_ax[0] + np.timedelta64(120, 's')
time_ax_range_max = time_ax[0] + np.timedelta64(300, 's')

# plot
fig, ax = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax[0].plot(B_64hz_fac_bandpass_da.time, B_64hz_fac_bandpass_da.sel(v_dim=0))
ax[0].set_ylabel('B_x [nT]')
ax[0].minorticks_on()
ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
#ax[0].set_ylim(-0.5, 0.5)
ax[1].plot(B_64hz_fac_bandpass_da.time, B_64hz_fac_bandpass_da.sel(v_dim=1))
ax[1].set_ylabel('B_y [nT]')
ax[1].minorticks_on()
ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
#ax[1].set_ylim(-0.5, 0.5)
ax[2].plot(B_64hz_fac_bandpass_da.time, B_64hz_fac_bandpass_da.sel(v_dim=2))
ax[2].set_ylabel('B_z [nT]')
ax[2].set_xlabel('Time')
ax[2].minorticks_on()
ax[2].grid(True, which='both', linestyle='--', alpha=0.5)
ax[2].set_xlim(time_ax_range_min, time_ax_range_max)
#ax[2].set_ylim(-0.5, 0.5)
plt.tight_layout()
plt.show()

## 垂直磁場成分の振幅$B_{\mathrm{w}}$と位相$\psi_{\mathrm{B}}$を取得したい

In [ ]:
B_64hz_fac_bandpass_perp_amp = np.sqrt(B_64hz_fac_bandpass_da.sel(v_dim=0)**2 + B_64hz_fac_bandpass_da.sel(v_dim=1)**2)
B_64hz_fac_bandpass_perp_phase = (np.degrees(np.arctan2(B_64hz_fac_bandpass_da.sel(v_dim=1), B_64hz_fac_bandpass_da.sel(v_dim=0)))) % 360.

B_64hz_fac_bandpass_perp_da = xr.DataArray(
    np.stack((B_64hz_fac_bandpass_perp_amp, B_64hz_fac_bandpass_perp_phase), axis=1),
    dims=['time', 'variable'],
    coords={
        'time': B_64hz_fac_bandpass_da.time,
        'variable': ['B_perp_amp', 'B_perp_phase']
    },
    name='erg_mgf_l2_mag_64hz_fac_bandpass_perp'
)
print(B_64hz_fac_bandpass_perp_da)

In [ ]:
time_ax_range_min = time_ax[0] + np.timedelta64(240, 's')
time_ax_range_max = time_ax[0] + np.timedelta64(300, 's')

fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax[0].plot(B_64hz_fac_bandpass_perp_da.time, B_64hz_fac_bandpass_perp_da.sel(variable='B_perp_amp'))
ax[0].set_ylabel('|B_perp| [nT]')
ax[0].set_title('Band-pass filtered |B_perp| (0.6-0.75 Hz)')
ax[0].minorticks_on()
ax[0].grid(True, which='both', linestyle='--', alpha=0.5)
ax[0].set_xlim(time_ax_range_min, time_ax_range_max)

ax[1].plot(B_64hz_fac_bandpass_perp_da.time, B_64hz_fac_bandpass_perp_da.sel(variable='B_perp_phase'))
ax[1].set_ylabel('Phase of B_perp [deg]')
ax[1].set_title('Band-pass filtered Phase of B_perp (0.6-0.75 Hz)')
ax[1].set_xlabel('Time')
ax[1].minorticks_on()
ax[1].grid(True, which='both', linestyle='--', alpha=0.5)
ax[1].set_xlim(time_ax_range_min, time_ax_range_max)
plt.tight_layout()
plt.show()

## $\zeta := \phi - \psi_{\mathrm{B}}$より、(time, (flux, pitch angle, zeta angle))のデータを作成

In [ ]:
flux_pitch_gyro_data_arrays

In [ ]:
flux_pitch_zeta_data_arrays = {}
for energy_i in range(energy_num):
    time_ax = flux_pitch_gyro_data_arrays[energy_i]['time'].values
    B_64hz_fac_bandpass_perp_da_interp = B_64hz_fac_bandpass_perp_da.interp(time=time_ax)
    flux_pitch_zeta_data_arrays[energy_i] = xr.DataArray(
        np.stack((
            flux_pitch_gyro_data_arrays[energy_i].data[:, :, 0],               # (TxC) flux
            flux_pitch_gyro_data_arrays[energy_i].data[:, :, 1],               # (TxC) pitch angle
            (flux_pitch_gyro_data_arrays[energy_i].data[:, :, 2] - B_64hz_fac_bandpass_perp_da_interp.sel(variable='B_perp_phase').values[:, np.newaxis]) % 360.   # (TxC) zeta angle
        ), axis=2),                                                           # (TxC, 3)
        dims=['time', 'channel', 'variable'],
        coords={
            'time':         flux_pitch_gyro_data_arrays[energy_i]['time'].values,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i],
            'variable':     ['flux', 'pitch_angle_deg', 'zeta_deg']
        },
        attrs=flux3d.attrs,
        name=f'flux_pitch_zeta_energy_{energy_i}'
    )
    print(f'energy {energy_i}: flux_pitch_zeta_data_arrays[{energy_i}]  = ', flux_pitch_zeta_data_arrays[energy_i])

In [ ]:
time_ax_range_min = time_ax[0]
time_ax_range_max = time_ax[0] + np.timedelta64(5, 'm')

for count_i in range(energy_num):
    time_ax = flux_pitch_zeta_data_arrays[count_i]['time'].values
    time_ax_mesh, channel_ax_mesh = np.meshgrid(time_ax, channel_ax, indexing='ij')
    pitch_angle_data = flux_pitch_zeta_data_arrays[count_i].data[:, :, 1]
    zetaphase_data = flux_pitch_zeta_data_arrays[count_i].data[:, :, 2]
    flux_data = flux_pitch_zeta_data_arrays[count_i].data[:, :, 0]

    if count_i != 5:
        continue

    # 横軸: 時間、縦軸: pitch angle, colorbar: fluxのplot
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from matplotlib.colors import LogNorm 
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    # 横軸: time_ax_mesh, 縦軸: pitch_angle_data, colorbar: flux_data
    sc = ax0.scatter(
        time_ax_mesh.flatten(),
        pitch_angle_data.flatten(),
        c=flux_data.flatten(),
        s=10,
        cmap='turbo',
        norm=LogNorm()
        )
    ax0.set_ylabel('pitch angle (deg)')
    ax0.set_title(f'LEP-i flux (energy = {energy_ax[count_i]:.4f} keV)')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.minorticks_on()
    ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 180)
    ax0.set_yticks(np.arange(0, 181, 15))
    ax0.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(sc, ax=ax0, label='flux [#/cm2/sr/sec/keV]')
    ax1 = fig.add_subplot(212)
    # 横軸: time_ax_mesh, 縦軸: zetaphase_data, colorbar: flux_data
    sc = ax1.scatter(
        time_ax_mesh.flatten(),
        zetaphase_data.flatten(),
        c=flux_data.flatten(),
        s=10,
        cmap='turbo',
        norm=LogNorm()
        )
    ax1.set_ylabel('zetaphase (deg)')
    ax1.set_title(f'LEP-i flux (energy = {energy_ax[count_i]:.4f} keV)')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.minorticks_on()
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360)
    ax1.set_yticks(np.arange(0, 361, 30))
    ax1.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(sc, ax=ax1, label='flux [#/cm2/sr/sec/keV]')
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import xarray as xr
from scipy.stats import binned_statistic_2d
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import matplotlib.colors as mcolors
from scipy.ndimage import convolve1d

pitch_angle_bins = np.linspace(0, 180, 17)    # 16 bins (0, 11.25, 22.5, ..., 180)
zeta_bins = np.linspace(0, 360, 13)           # 14 bins (0, 30, 60, ..., 360)
dt = 4.0

time_bins = np.arange(time_ax_range_min.astype('datetime64[s]').astype('float'),
                      time_ax_range_max.astype('datetime64[s]').astype('float') + dt,
                      dt)   # dt secごとのbin
time_bins = time_bins.astype('datetime64[s]')
print('time_bins = ', time_bins)

def to_sec_i64(t):
    t = np.asarray(t).astype('datetime64[s]')
    return t.astype('int64')  # epoch秒

def make_lognorm(arr, lo_q=0, hi_q=100, min_span=10.0):
    # 正の有限値だけ
    pos = arr[np.isfinite(arr) & (arr > 0)]
    if pos.size == 0:
        return None
    vmin = np.nanpercentile(pos, lo_q)
    vmax = np.nanpercentile(pos, hi_q)
    # 幅が潰れたら最低幅を確保
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin * min_span
    return mcolors.LogNorm(vmin=vmin, vmax=vmax)

def make_linearnorm(arr, lo_q=0, hi_q=100, min_span=0.1):
    # 有限値だけ
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return None
    vmin = np.nanpercentile(finite, lo_q)
    vmax = np.nanpercentile(finite, hi_q)
    # 幅が潰れたら最低幅を確保
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + min_span
    #return mcolors.Normalize(vmin=vmin, vmax=vmax)
    return mcolors.Normalize(vmin=0.6, vmax=1.0)

def movmean_time_nan(A, win):
    W = np.ones(int(win), dtype=float)
    valid = np.isfinite(A)
    A0 = np.where(valid, A, 0.0)
    num = convolve1d(A0,    W, axis=0, mode='constant', cval=0.0)      # 値の和
    den = convolve1d(valid.astype(float), W, axis=0, mode='constant', cval=0.0)  # 有効数
    out = num / den
    out[den == 0] = np.nan
    return out

# 時間ビン（秒の整数エッジに）
time_bins_sec = to_sec_i64(time_bins)
for energy_i in range(energy_num):
    if energy_i != 5:
        continue
    time_ax = flux_pitch_zeta_data_arrays[energy_i]['time'].values
    pitch_angle_data = flux_pitch_zeta_data_arrays[energy_i].data[:, :, 1]
    zeta_data       = flux_pitch_zeta_data_arrays[energy_i].data[:, :, 2]
    flux_data       = flux_pitch_zeta_data_arrays[energy_i].data[:, :, 0]
    
    # 時刻を2Dに展開 → 秒の整数に
    time_ax_mesh = np.broadcast_to(time_ax[:, None], pitch_angle_data.shape)
    tx_sec = to_sec_i64(time_ax_mesh.ravel())

    pa = pitch_angle_data.ravel().astype(float)
    ze = zeta_data.ravel().astype(float)
    fv = flux_data.ravel().astype(float)

    good = np.isfinite(pa) & np.isfinite(ze) & np.isfinite(fv)
    tx_sec, pa, ze, fv = tx_sec[good], pa[good], ze[good], fv[good]

    # --- 2Dビニング（平均：欠損は無視） ---
    hist_pitch, xedges_pitch, yedges_pitch, _ = binned_statistic_2d(
        tx_sec, pa, fv, statistic=np.nanmean,
        bins=[time_bins_sec, pitch_angle_bins]
    )
    hist_zeta, xedges_zeta, yedges_zeta, _ = binned_statistic_2d(
        tx_sec, ze, fv, statistic=np.nanmean,
        bins=[time_bins_sec, zeta_bins]
    )

    # 60秒で移動平均
    hist_pitch = movmean_time_nan(hist_pitch, 15)
    hist_zeta = movmean_time_nan(hist_zeta, 15)


    # fluxは最大値で正規化
    hist_pitch /= np.nanmax(hist_pitch)
    hist_zeta  /= np.nanmax(hist_zeta)

    # ビン中心 → 描画用に datetime64 へ戻す
    tcent_sec = (xedges_pitch[:-1] + xedges_pitch[1:]) // 2
    time_cent = tcent_sec.astype('datetime64[s]')
    pa_cent   = 0.5*(yedges_pitch[:-1] + yedges_pitch[1:])
    ze_cent   = 0.5*(yedges_zeta[:-1]  + yedges_zeta[1:])

    cnt_pitch, _, _ = np.histogram2d(tx_sec, pa, bins=[time_bins_sec, pitch_angle_bins])
    cnt_zeta, _, _  = np.histogram2d(tx_sec, ze, bins=[time_bins_sec, zeta_bins])
    hist_pitch[cnt_pitch == 0] = np.nan
    hist_zeta[cnt_zeta == 0]   = np.nan
    #norm_pitch = make_lognorm(hist_pitch)
    #norm_zeta  = make_lognorm(hist_zeta)
    norm_pitch = make_linearnorm(hist_pitch)
    norm_zeta  = make_linearnorm(hist_zeta)
    if norm_pitch is None or norm_zeta is None:
        print(f'energy {energy_i}: no positive flux data')
        continue
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    pcm = ax0.pcolormesh(
        time_cent, pa_cent, hist_pitch.T,
        cmap='turbo', norm=norm_pitch,
        shading='auto'
    )
    ax0.set_ylabel('pitch angle (deg)')
    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.minorticks_on()
    ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 180); ax0.set_yticks(np.arange(0, 181, 15))
    ax0.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(pcm, ax=ax0)#, label='flux [#/cm2/sr/sec/keV]')
    ax1 = fig.add_subplot(212)
    pcm = ax1.pcolormesh(
        time_cent, ze_cent, hist_zeta.T,
        cmap='turbo', norm=norm_zeta,
        shading='auto'
    )
    ax1.set_ylabel('zeta (deg)')
    ax1.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.minorticks_on()
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360); ax1.set_yticks(np.arange(0, 361, 30))
    ax1.set_xlim(time_ax_range_min, time_ax_range_max)
    plt.colorbar(pcm, ax=ax1)#, label='flux [#/cm2/sr/sec/keV]')
    plt.tight_layout()
    plt.show()

In [ ]:
# 上図: Bw amplitude-time, 下図: zeta-time spectrogram (125 < pitch angle < 145 deg)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import Normalize
import matplotlib.colors as mcolors
from scipy.stats import binned_statistic_2d
import xarray as xr

zeta_bins = np.linspace(0, 360, 13)           # 14 bins (0, 30, 60, ..., 360)
dt = 4.0
time_bins = np.arange(time_ax_range_min.astype('datetime64[s]').astype('float') + 4.0,
                      time_ax_range_max.astype('datetime64[s]').astype('float') + dt + 4.0,
                      dt)   # dt secごとのbin
time_bins = time_bins.astype('datetime64[s]')

def to_sec_i64(t):
    t = np.asarray(t).astype('datetime64[s]')
    return t.astype('int64')  # epoch秒

def make_linearnorm(arr, lo_q=0, hi_q=100, min_span=0.1):
    return mcolors.Normalize(vmin=0.6, vmax=1.0)

# 時間ビン（秒の整数エッジに）
time_bins_sec = to_sec_i64(time_bins)

for energy_i in range(energy_num):
    if energy_i != 5:
        continue
    time_ax_1 = flux_pitch_zeta_data_arrays[energy_i]['time'].values
    pa2d = flux_pitch_zeta_data_arrays[energy_i].data[:, :, 1]
    ze2d = flux_pitch_zeta_data_arrays[energy_i].data[:, :, 2]
    fv2d = flux_pitch_zeta_data_arrays[energy_i].data[:, :, 0]

    # ピッチ角マスク（要素ごと）
    mask_pa = (pa2d >= 125) & (pa2d <= 145)

    # 時刻を2Dに展開して“同じマスク”で抜き出す
    t2d = np.broadcast_to(time_ax_1[:, None], pa2d.shape)     # (Nt, Nch)
    tx_sec = to_sec_i64(t2d[mask_pa])                         # 1D
    ze = ze2d[mask_pa].astype(float)                          # 1D
    fv = fv2d[mask_pa].astype(float)                          # 1D

    # 欠損除去（全配列同じ長さ）
    good = np.isfinite(tx_sec) & np.isfinite(ze) & np.isfinite(fv)
    tx_sec, ze, fv = tx_sec[good], ze[good], fv[good]

    # 2Dビニング（平均）
    hist_zeta, xedges_zeta, yedges_zeta, _ = binned_statistic_2d(
        tx_sec, ze, fv, statistic=np.nanmean,
        bins=[time_bins_sec, zeta_bins]
    )

    hist_zeta = movmean_time_nan(hist_zeta, 15)

    # サンプル不足ビンを NaN
    cnt_zeta, _, _ = np.histogram2d(tx_sec, ze, bins=[time_bins_sec, zeta_bins])
    hist_zeta[cnt_zeta == 0] = np.nan

    # 全NaN/ゼロ回避して正規化
    pos = hist_zeta[np.isfinite(hist_zeta)]
    if pos.size == 0:
        print(f'energy {energy_i}: no data in PA 125–145'); continue
    hist_zeta /= np.nanmax(pos)

    # ビン中心（描画用）
    tcent_sec = (xedges_zeta[:-1] + xedges_zeta[1:]) // 2
    time_cent = tcent_sec.astype('datetime64[s]')
    ze_cent   = 0.5*(yedges_zeta[:-1] + yedges_zeta[1:])
    cnt_zeta, _, _  = np.histogram2d(tx_sec, ze, bins=[time_bins_sec, zeta_bins])
    hist_zeta[cnt_zeta == 0]   = np.nan
    norm_zeta  = make_linearnorm(hist_zeta)
    if norm_zeta is None:
        print(f'energy {energy_i}: no positive flux data')
        continue

    gs = plt.figure(figsize=(10,6)).add_gridspec(2, 20, hspace=0.05)
    ax0 = plt.gcf().add_subplot(gs[0, :19])
    ax0.plot(B_64hz_fac_bandpass_perp_da.time, B_64hz_fac_bandpass_perp_da.sel(variable='B_perp_amp'), lw=4, c='k')
    ax0.set_ylabel(r'$|\bf{B}_{\mathrm{w}}|$' + '\n' + '[nT]')
    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV), 125 < PA < 145')
    ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax0.minorticks_on()
    ax0.grid(True, which='both', linestyle='--', alpha=0.5)
    ax0.set_ylim(0, 10); ax0.set_yticks(np.arange(0, 10, 1))
    ax0.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.tick_params(labelbottom=False)

    ax1 = plt.gcf().add_subplot(gs[1, :19], sharex=ax0)
    cax = plt.gcf().add_subplot(gs[:, 19])
    pcm = ax1.pcolormesh(
        time_cent, ze_cent, hist_zeta.T,
        cmap='jet', norm=norm_zeta,
        shading='auto'
    )
    ax1.set_ylabel(r'$\zeta$' + '\n' + r'[deg]')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax1.minorticks_on()
    ax1.grid(True, which='both', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 360); ax1.set_yticks(np.arange(0, 361, 30))
    ax1.set_xlim(time_ax_range_min, time_ax_range_max)

    plt.colorbar(pcm, cax=cax)#, label='flux [#/cm2/sr/sec/keV]')
    plt.tight_layout()
    plt.show()